# Data checks — 2025

Review the station and price tables produced by `src/parse_data.py` from the downloaded archives. These checks identify issues before building the main diesel panel; they do not modify the parquet files.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

YEAR = 2025
# Support execution from either the project root or the notebook directory.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data/interim").is_dir() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "interim"

stations = pd.read_parquet(DATA_DIR / f"stations_{YEAR}.parquet")
prices = pd.read_parquet(DATA_DIR / f"prices_{YEAR}.parquet")

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 20)
display(stations.head())
display(prices.head())

,station_id,source_year,latitude,longitude,postal_code,road_type,address,city,automate_24_24,n_services,services
0,1000001,2025,46.201000,5.19800,01000,R,596 AVENUE DE TREVOUX,SAINT-DENIS-LèS-BOURG,False,3,DAB (Distributeur automatique de billets) | St...
1,1000002,2025,46.218420,5.22767,01000,R,16 Avenue de Marboz,BOURG-EN-BRESSE,False,2,DAB (Distributeur automatique de billets) | Ve...
2,1000004,2025,46.188000,5.24500,01000,R,20 Avenue du Maréchal Juin,Bourg-en-Bresse,True,13,Automate CB 24/24 | Boutique alimentaire | Bou...
3,1000005,2025,46.200936,5.19942,01000,R,642 Avenue de Trévoux,SAINT-DENIS-LèS-BOURG,False,8,Boutique alimentaire | Boutique non alimentair...
4,1000006,2025,46.207540,5.23758,01000,R,1 Boulevard John Kennedy,BOURG-EN-BRESSE,False,1,"Vente de gaz domestique (Butane, Propane)"


,station_id,fuel_id,fuel,timestamp,price,source_year
0,1000001,1,Gazole,2025-01-02 00:37:00,1.707,2025
1,1000001,1,Gazole,2025-01-02 07:56:00,1.707,2025
2,1000001,1,Gazole,2025-01-03 00:38:00,1.725,2025
3,1000001,1,Gazole,2025-01-07 00:37:00,1.705,2025
4,1000001,1,Gazole,2025-01-09 00:37:00,1.688,2025


## Dataset overview

Check table sizes, missing values and the number of observations per fuel.

In [2]:
display(pd.DataFrame(
    {"rows": [len(stations), len(prices)],
     "columns": [stations.shape[1], prices.shape[1]]},
    index=["Stations", "Prices"],
))
display(prices.isna().sum().rename("missing_values").to_frame())
display(prices["fuel"].value_counts(dropna=False).rename("observations").to_frame())

,rows,columns
Stations,14420,11
Prices,5302458,6


,missing_values
station_id,0
fuel_id,4658
fuel,4658
timestamp,4658
price,4658
source_year,0


,observations
fuel,
Gazole,1520477
E10,1351304
SP98,1291207
E85,581219
SP95,281659
GPLc,271934
NaN,4658


## Unique station identifiers

Each station must have one nonmissing `station_id` and appear only once in the station table.

In [3]:
repeated_station_ids = stations["station_id"].duplicated(keep=False)
station_id_check = pd.Series({
    "Station rows": len(stations),
    "Unique station IDs": stations["station_id"].nunique(),
    "Missing station IDs": stations["station_id"].isna().sum(),
    "Rows with repeated station IDs": repeated_station_ids.sum(),
}, name="count")
display(station_id_check.to_frame())
display(pd.Series({
    "One row per station, with no missing ID": (
        stations["station_id"].notna().all()
        and stations["station_id"].is_unique
    )
}, name="passed").to_frame())
if repeated_station_ids.any():
    display(stations.loc[repeated_station_ids].sort_values("station_id").head(20))

,count
Station rows,14420
Unique station IDs,14420
Missing station IDs,0
Rows with repeated station IDs,0


,passed
"One row per station, with no missing ID",True


## Dates

Use the observation timestamp to check the study year. Missing timestamps are counted separately from dates outside the year.

In [4]:
year_start = pd.Timestamp(YEAR, 1, 1)
year_end = pd.Timestamp(YEAR + 1, 1, 1)
in_year = prices["timestamp"].ge(year_start) & prices["timestamp"].lt(year_end)
outside_year = prices["timestamp"].notna() & ~in_year

out_of_year_observations = prices.loc[outside_year].sort_values("timestamp")
display(pd.Series({
    "Observations within the study year": in_year.sum(),
    "Observations before the study year": prices["timestamp"].lt(year_start).sum(),
    "Observations after the study year": prices["timestamp"].ge(year_end).sum(),
    "Missing timestamps": prices["timestamp"].isna().sum(),
}, name="count").to_frame())
display(prices["timestamp"].agg(["min", "max"]).rename("timestamp").to_frame())
if not out_of_year_observations.empty:
    display(out_of_year_observations.head(10))
    display(out_of_year_observations.tail(10))

,count
Observations within the study year,5297800
Observations before the study year,0
Observations after the study year,0
Missing timestamps,4658


,timestamp
min,2025-01-01 00:00:00
max,2025-12-31 23:59:00


## Duplicate price observations

The observation key is `(station_id, fuel, timestamp)`. For repeated keys, compare prices to distinguish identical observations from conflicting reports. Incomplete keys and missing prices are reported separately.

In [5]:
price_key = ["station_id", "fuel", "timestamp"]
complete_key = prices[price_key].notna().all(axis=1)
duplicate_prices = prices.loc[
    complete_key & prices.duplicated(price_key, keep=False)
].copy()

duplicate_groups = duplicate_prices.groupby(price_key).agg(
    rows=("price", "size"),
    distinct_prices=("price", "nunique"),
    missing_prices=("price", lambda values: values.isna().sum()),
    prices_seen=("price", lambda values: tuple(sorted(values.dropna().unique()))),
)
same_price = duplicate_groups["distinct_prices"].eq(1) & duplicate_groups["missing_prices"].eq(0)
conflicting_price = duplicate_groups["distinct_prices"].gt(1)
unresolved_price = ~(same_price | conflicting_price)

display(pd.Series({
    "Rows with incomplete keys (excluded from comparison)": (~complete_key).sum(),
    "Rows in repeated-key groups": len(duplicate_prices),
    "Repeated-key groups": len(duplicate_groups),
    "Groups with identical, nonmissing prices": same_price.sum(),
    "Groups with conflicting prices": conflicting_price.sum(),
    "Groups unresolved because of missing prices": unresolved_price.sum(),
    "Groups containing any missing price": duplicate_groups["missing_prices"].gt(0).sum(),
}, name="count").to_frame())
if not duplicate_groups.empty:
    display(duplicate_groups.assign(
        status=np.select(
            [conflicting_price, same_price],
            ["Conflicting prices", "Identical prices"],
            default="Missing price: review needed",
        )
    ).sort_values(["status", "rows"], ascending=[True, False]).head(20))

,count
Rows with incomplete keys (excluded from comparison),4658
Rows in repeated-key groups,2
Repeated-key groups,1
"Groups with identical, nonmissing prices",0
Groups with conflicting prices,1
Groups unresolved because of missing prices,0
Groups containing any missing price,0


,,,rows,distinct_prices,missing_prices,prices_seen,status
station_id,fuel,timestamp,,,,,
24650001,E10,2025-09-05 08:33:38,2,2,0,"(1.56, 1.566)",Conflicting prices


## Minimum and maximum prices

Inspect ranges by fuel and retain the station and timestamp of each extreme. Unusual values require review before exclusion; the price distribution alone cannot establish a reporting or parsing error.

In [6]:
finite_price = prices["price"].notna() & np.isfinite(prices["price"])
finite_prices = prices.loc[finite_price]
price_summary = finite_prices.groupby("fuel")["price"].agg(
    observations="count",
    minimum="min",
    p01=lambda values: values.quantile(0.01),
    median="median",
    p99=lambda values: values.quantile(0.99),
    maximum="max",
)
display(price_summary.round(3))
display(pd.Series({
    "Missing prices": prices["price"].isna().sum(),
    "Infinite prices": (prices["price"].notna() & ~finite_price).sum(),
    "Nonpositive prices": prices["price"].le(0).sum(),
}, name="count").to_frame())

by_fuel = finite_prices.groupby("fuel")["price"]
extreme_prices = pd.concat([
    prices.loc[by_fuel.idxmin()].assign(extreme="Minimum"),
    prices.loc[by_fuel.idxmax()].assign(extreme="Maximum"),
]).sort_values(["fuel", "price"])
# One representative row for each minimum/maximum, including when there are ties.
display(extreme_prices[["fuel", "extreme", "price", "station_id", "timestamp"]])

,observations,minimum,p01,median,p99,maximum
fuel,,,,,,
E10,1351304,1.261,1.549,1.699,2.014,2.664
E85,581219,0.385,0.630,0.749,0.999,1.999
GPLc,271934,0.496,0.829,0.985,1.189,1.989
Gazole,1520477,1.213,1.473,1.636,1.949,3.000
SP95,281659,1.326,1.586,1.717,1.910,2.889
SP98,1291207,1.296,1.639,1.796,2.154,2.997


,count
Missing prices,4658
Infinite prices,0
Nonpositive prices,0


,fuel,extreme,price,station_id,timestamp
567475,E10,Minimum,1.261,13630004,2025-04-18 10:03:28
658184,E10,Maximum,2.664,15100010,2025-07-15 11:04:26
733666,E85,Minimum,0.385,17690001,2025-09-25 08:47:17
65985,E85,Maximum,1.999,2190003,2025-12-15 06:00:00
199558,GPLc,Minimum,0.496,6210010,2025-06-26 06:07:07
1594933,GPLc,Maximum,1.989,34500008,2025-01-14 11:13:08
304990,Gazole,Minimum,1.213,9000004,2025-04-18 07:38:34
4785590,Gazole,Maximum,3.000,90150001,2025-06-04 18:41:05
1049084,SP95,Minimum,1.326,27120001,2025-07-07 09:34:22
1554506,SP95,Maximum,2.889,34110004,2025-01-25 08:13:01


## Stations offering diesel (Gazole)

Count stations with at least one dated, positive, finite Gazole price in the study year. This gives the candidate population for the main panel; continuous coverage still needs checking.

In [7]:
is_diesel = prices["fuel"].eq("Gazole")
diesel_in_year = is_diesel & in_year
diesel_candidates = diesel_in_year & finite_price & prices["price"].gt(0)
diesel_station_ids = pd.Index(
    prices.loc[diesel_candidates, "station_id"].dropna().unique(),
    name="station_id",
).sort_values()
missing_diesel_metadata = diesel_station_ids.difference(stations["station_id"].dropna())

display(pd.Series({
    "Stations with any Gazole record in the archive": prices.loc[is_diesel, "station_id"].nunique(),
    "Stations with a Gazole observation in the study year": prices.loc[diesel_in_year, "station_id"].nunique(),
    "Candidate diesel stations with a positive, finite price": len(diesel_station_ids),
    "Candidate diesel stations missing from the station table": len(missing_diesel_metadata),
}, name="stations").to_frame())

,stations
Stations with any Gazole record in the archive,9712
Stations with a Gazole observation in the study year,9712
"Candidate diesel stations with a positive, finite price",9712
Candidate diesel stations missing from the station table,0


## Findings

- All **14,420 station IDs are unique and nonmissing**. All **5,297,800 dated observations** fall in 2025; **4,658 rows** lack fuel, timestamp and price.
- One repeated key has conflicting E10 prices: station `24650001`, `2025-09-05 08:33:38`, **1.560 vs 1.566**. No repeated key has identical nonmissing prices.
- Prices range from **0.385 to 3.000 EUR/L** overall, with no nonpositive or infinite values. Spot checks in the raw 2025 XML confirm the E85 minimum, Gazole maximum and E10 conflict already exist in the source; possible reporting errors still need review.
- **9,712 stations** have at least one positive Gazole price in 2025 and all have station metadata. This is the starting population for the main panel.